# TP1 — Problem Settings and Data Generation  
## Inverse Parametrized Poisson Problem on a 3D Rock

This notebook defines the **problem settings** for **Test Problem 1 (TP1)**, which addresses an **inverse parametrized Poisson problem** on a complex **3D rock geometry**.

The purpose of this notebook is to:
- define the physical problem and its parameters,
- generate **simulated IoT-like boundary measurements**,
- acquire and preprocess the 3D geometry,
- produce the mesh and visualization-ready files required by the subsequent stages.

This notebook represents the **first stage of the TP1 pipeline** and prepares all the assets used in the offline, inverse, and online stages.

In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models", "rock_poisson_inverse", "trainPOD"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

## Libraries and Dependencies

We start by importing all the libraries required for:
- geometry handling and mesh generation,
- data management,
- preparation of visualization files.

In [ ]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

## Numerical Precision

Double precision is enforced

In [ ]:
torch.set_default_dtype(torch.float64)

## Parametrized Physical Problem

We consider a 3D domain $\Omega \subseteq \mathbb{R}^3$ representing a **rock geometry**, with boundary $\Gamma = \partial \Omega$.  
The physical phenomenon is described by the following differenzial problem:
$$
\begin{equation}
    \begin{cases}
        \frac{\partial^2 u}{\partial x^2} \left(x, y, z\right) + \frac{\partial^2 u}{\partial y^2} \left(x, y, z \right) + \frac{\partial^2 u}{\partial z^2} \left(x, y, z\right) = - \left( \alpha^2 + \beta^2 \right) \pi^2 \lambda x \cos\left( \alpha \pi y\right) \sin\left( \beta \pi z\right) & \left(x, y, z \right) \in \Omega \\
        u \left( x, y, z \right) = \lambda x \cos\left( \alpha \pi y \right) \sin \left( \beta \pi z \right) & \left( x, y, z \right) \in \Gamma
    \end{cases}
    \tag{1}
\end{equation}
$$

The scalar field u depends on a set of unknown parameters $\mu = \left( \lambda, \alpha, \beta \right)$.

The following analytical solution
$$
\begin{equation}
    u \left( x, y, z \right) = \lambda x \cos\left( \alpha \pi y \right) \sin \left( \beta \pi z \right) \qquad \left( x, y, z \right) \in \bar{\Omega}
    \tag{2}
\end{equation}
$$
is available and is used to generate synthetic measurements.

## Generation of Data for the Inverse Problem

In this stage, we generate **simulated measurements** that emulate IoT sensor data.

Sensors are assumed to be located on the boundary of the 3D domain.  
The corresponding values of the physical field are computed using the analytical solution and stored for later use in the inverse PINN stage.

In [ ]:
rock = Blend2Pina(LOAD_MODEL + model_name)

## Boundary Sampling

In [ ]:
num_points = 500

surface = rock.boundary()
points = surface.sample(num_points)

## Parameter Selection and Simulated Measurements

A reference set of parameters $\mu = \left( \lambda, \alpha, \beta \right)$ is fixed to generate the synthetic data.

Using the analytical solution:
- boundary values are computed at the selected sensor locations,
- the resulting measurements are stored in tabular form,
- these data will later be used as observations in the inverse PINN problem.

In [ ]:
par_lambda = .1
par_alpha = .2
par_beta = .5

u = par_lambda * points.extract("x").tensor * torch.cos(par_alpha * torch.pi * points.extract("y").tensor) * torch.sin(par_beta * torch.pi * points.extract("z").tensor)

## Data Storage

The simulated measurements and sensor locations are saved in CSV format.

This ensures:
- reproducibility of the inverse problem,
- decoupling between data generation and inference,
- reuse of the same dataset across multiple experiments.

In [ ]:
total_info = torch.concat(
    [points.tensor, u],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "u"]
)

df.to_csv("./files/data.csv", sep=";")

## Mesh Generation and Visualization Files

The 3D rock geometry is discretized to generate a computational mesh suitable for numerical simulations.

In addition:
- visualization-ready files (.xdmf and associated data) are produced,
- these files enable inspection of solutions and errors in ParaView,
- the same mesh is reused consistently across all TP1 stages.

In [ ]:
rock_msh = Blend2Mesh(LOAD_MODEL + model_name, "rock")

In [ ]:
rock_msh.create_mesh(len_msh=0.07)

In [ ]:
rock_xdmf = Msh2Xdmf("rock.msh", "rock")
rock_xdmf.to_xdmf()